In [2]:
# extracted data from lidl plus receipt
receipt = [
    "Banane Fair Bio",
    "Zwiebeln weiß",
    "Bioland Zucchini",
    "Petersilie",
    "Bio Limetten",
    "Weidemilch 3,8%",
    "Exquisa fitline Natur",
    "Bio-Eier OKT 10er",
    "Bio Müsli Schoko",
    "Lava Bar Cookie",
    "Dinkelschnitte",
    "Walnusskerne",
    "Bio Cashewkerne",
    "Rinderhackfleisch 5%",
    "Puten-Geschnetzeltes"
]

In [4]:
!pip install requests pandas

In [5]:
import requests
import pandas as pd
from time import sleep

BASE_URL = "https://world.openfoodfacts.org/cgi/search.pl"

headers = {
    "User-Agent": "NutritionOptimizerPrototype/0.1 (student project)"
}

def search_product(product_name):

    params = {
        "search_terms": product_name,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": 3
    }

    response = requests.get(
        BASE_URL,
        params=params,
        headers=headers
    )

    data = response.json()

    return data.get("products", [])

for item in receipt:

    print("="*60)
    print(item)

    products = search_product(item)

    print(f"{len(products)} matches")

    for p in products:

        print(
            p.get("product_name"),
            "|",
            p.get("brands"),
            "|",
            p.get("countries")
        )

Banane Fair Bio
3 matches
Barres céréales Banane Chocolat noir | Bio Soleil | France
Banane BIO | Intermarché, Mon Marché plaisir | France
Banane Joghurt | Migros Bio | 
Zwiebeln weiß


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [6]:
def search_product(product_name):

    params = {
        "search_terms": product_name,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": 3,
        "fields": "product_name,brands,countries,nutriscore_grade"
    }

    response = requests.get(BASE_URL, params=params, headers=headers, timeout=10)
    response.raise_for_status()

    data = response.json()

    results = []

    for p in data.get("products", []):
        results.append({
            "query": product_name,
            "product_name": p.get("product_name"),
            "brands": p.get("brands"),
            "countries": p.get("countries"),
            "nutriscore": p.get("nutriscore_grade")
        })

    return results

In [7]:
headers = {
    "User-Agent": "NutritionOptimizerPrototype/0.1 (student project)",
    "Accept": "application/json"
}

search_product("Plattpfirsiche")

HTTPError: 503 Server Error: Service Temporarily Unavailable for url: https://world.openfoodfacts.org/cgi/search.pl?search_terms=Plattpfirsiche&search_simple=1&action=process&json=1&page_size=3&fields=product_name%2Cbrands%2Ccountries%2Cnutriscore_grade

In [ ]:
# add "germany" as country to better match products

In [14]:
import requests

def get_product_by_ean(ean: str):
    url = f"https://world.openfoodfacts.org/api/v0/product/{ean}.json"
    
    r = requests.get(url)
    data = r.json()
    
    if data.get("status") != 1:
        return None  # product not found
    
    product = data["product"]
    
    return {
        "barcode": product.get("code"),
        "name": product.get("product_name"),
        "brand": product.get("brands"),
        "ingredients": product.get("ingredients_text"),
        "nutriscore": product.get("nutriscore_grade"),
        "nova_group": product.get("nova_group"),
        "categories": product.get("categories")
    }


# example
print(get_product_by_ean("9920758"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)